In [1]:
pip install pandas scikit-learn joblib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
import joblib

# 1. Load your dataset (Replace with your actual file path)
# Assuming a CSV with 'text' and 'label' columns
#try:
    #df = pd.read_csv('emails.csv') 
    #df = pd.read_csv('emails.csv', encoding='ISO-8859-1')
# Try loading with 'latin1' or 'iso-8859-1' encoding
try:
    df = pd.read_csv('emails.csv', encoding='latin1') 
except UnicodeDecodeError:
    # Backup option if latin1 fails
    df = pd.read_csv('emails.csv', encoding='cp1252')

except FileNotFoundError:
    # Creating a dummy dataframe so the code is immediately runnable for you
    data = {
        'text': [
            'Hey, are we still meeting for lunch today?',
            'CONGRATULATIONS! You have won a $1000 Walmart gift card. Click here to claim.',
            'Can you send me the report by 5 PM?',
            'URGENT: Your account has been compromised. Verify your password now at fake-link.com',
            'Dear professor, I will be late to class today.'
        ],
        'label': ['not spam', 'spam', 'not spam', 'spam', 'not spam']
    }
    df = pd.DataFrame(data)

# 2. Split data into features (X) and target labels (y)
X = df['text']
y = df['label']

# 3. Split into Training and Testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Feature Extraction: Convert text data into numerical vectors using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', lowercase=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 5. Train the Model (Multinomial Naive Bayes)
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# 6. Evaluate the Model
y_pred = model.predict(X_test_tfidf)
print("--- Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred))

# 7. Save the trained model and vectorizer for Real-World Implementation
joblib.dump(model, 'spam_detector_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("Model and Vectorizer saved successfully!")

--- Model Evaluation ---
Accuracy: 96.68%

              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       965
        spam       1.00      0.75      0.86       150

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115

Model and Vectorizer saved successfully!


In [ ]:
import joblib

def classify_email(email_text):
    # 1. Load the saved model and vectorizer
    loaded_model = joblib.load('spam_detector_model.pkl')
    loaded_vectorizer = joblib.load('tfidf_vectorizer.pkl')
    
    # 2. Preprocess and vectorize the incoming email
    email_vectorized = loaded_vectorizer.transform([email_text])
    
    # 3. Get the probability score first
    probabilities = loaded_model.predict_proba(email_vectorized)[0]
    spam_prob = probabilities[1] if loaded_model.classes_[1] == 'spam' else probabilities[0]
    
    # 4. Set a stricter threshold (e.g., 0.75) and manually predict
    threshold = 0.70
    prediction = 'spam' if spam_prob >= threshold else 'non-spam'
    
    return prediction, spam_prob

# --- Test it out ---
new_email = "Double your income overnight! Click this link to join our exclusive crypto matrix masterclass!"
label, confidence = classify_email(new_email)

print(f"Result: The email is classified as [{label.upper()}] (Spam Probability: {confidence*100:.2f}%)")

Result: The email is classified as [NON-SPAM] (Spam Probability: 40.35%)
